# Adversarial magnetization pairs: walkthrough

This notebook walks through the full workflow of *Topological ambiguity in stray-field magnetometry*
using the `adversarial_pairs` package:

1. **Set up** the geometry (grid, sample size, film thickness, standoff height, Ms) and optimizer settings.
2. **Choose two textures** with different topological charge, e.g. a skyrmion (Q = −1) and a meron (Q = −½).
3. **Optimize the pair** jointly so their stray fields agree while each charge is held at its target.
4. **Analyze** why the field cannot tell them apart: multipole moments and k-space localization.
5. **Validate statistically** with a Monte Carlo ensemble of random texture pairs, across standoff heights.
6. **Check sensitivity** to the grid resolution and standoff height.

The loss minimized for each pair is

$$\mathcal{L} = \mathrm{NRMSE}(\mathbf B_1,\mathbf B_2)^2 + \beta\,E_\text{topo} + \gamma\,E_\text{smooth},$$

where $E_\text{topo} = (Q_1-Q_1^*)^2 + (Q_2-Q_2^*)^2$ (anchor mode) or $(|Q_1-Q_2|-q)^2$ (difference mode),
and $\beta$ grows by penalty continuation until the charge constraint is met.
The stray field uses the exact thin-film FFT kernel; charges use the Berg–Lüscher solid-angle formula.

Every option shown here is also available from the command line (`python -m adversarial_pairs --help`).

## 0. Setup

In [ ]:
# On Colab, install the package first:
# !pip install -q git+https://github.com/shirso22/adversarial_magnetic_pairs.git
#
# Locally, from the repository root:  pip install -e .

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import torch

import adversarial_pairs as ap
from adversarial_pairs import (Geometry, OptimizerSettings, MonteCarloConfig,
                               generate_pair, make_texture, topological_charge,
                               stray_field, run_monte_carlo, standoff_sweep,
                               analysis, plotting)

print("adversarial_pairs", ap.__version__, "| torch", torch.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Geometry and optimizer settings

All physical inputs live in one `Geometry` object (lengths in nm, Ms in A/m).
The optimizer knobs live in `OptimizerSettings`. The most important choice is `ms_mode`:

| `ms_mode` | Meaning |
|---|---|
| `"free"` | both saturation magnetizations are optimized (bounded to `ms_bounds` × Ms0) |
| `"fixed"` | both held at the initial Ms — the strictest version of the problem |
| `"first"` / `"second"` | only texture 1 / texture 2 may change its Ms |

Defaults reproduce the manuscript's Sec. 5 test case.

In [ ]:
geom = Geometry(
    grid=(32, 32),            # cells (nx, ny)
    sample_size_nm=(64, 64),  # physical extent -> 2 nm cells
    thickness_nm=1.0,         # film thickness
    height_nm=20.0,           # standoff height of the sensor plane
    Ms=1e6,                   # initial saturation magnetization (A/m)
)

settings = OptimizerSettings(
    n_iter=450,        # Adam iterations
    lr=1e-2,
    gamma=1e-3,        # smoothness weight
    beta0=1.0,         # initial topology weight (grows x1.5 every 100 its while violated)
    ms_mode="free",    # "free" | "fixed" | "first" | "second"
    ms_bounds=(1.0, 5.0),
    log_every=150,
)
print(geom)
print(f"cell size {geom.dx_nm:.2f} x {geom.dy_nm:.2f} nm")

## 2. Textures

Texture classes are registered by name. Parameters can be passed as keyword arguments, or as a spec string
`"name:key=value,..."` (the same syntax the CLI uses). `topological_charge` uses the Berg–Lüscher
formula, so canonical textures come out exactly quantized.

In [ ]:
print(list(ap.TEXTURES))

specs = ["skyrmion", "meron", "fractional:charge=0.25", "skyrmion:helicity=0", "antiskyrmion", "uniform:phi_deg=30"]
fig, axes = plt.subplots(1, len(specs), figsize=(3 * len(specs), 3.2))
for ax, spec in zip(axes, specs):
    name, params = ap.parse_texture(spec)
    m = make_texture(name, geom.grid, **params)
    ax.imshow(m[..., 2].T.numpy(), origin="lower", cmap="RdBu_r", vmin=-1, vmax=1)
    s = 3
    ax.quiver(np.arange(0, geom.nx, s), np.arange(0, geom.ny, s),
              m[::s, ::s, 0].T.numpy(), m[::s, ::s, 1].T.numpy(), color="k", scale=25)
    ax.set_title(f"{spec}\nQ = {topological_charge(m).item():+.3f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("$m_z$ (colour) and in-plane m (arrows)")
plt.tight_layout()

Before optimization, two textures of different charge have very different stray fields.
The NRMSE below is the quantity the optimizer drives down.

In [ ]:
m1 = make_texture("skyrmion", geom.grid)
m2 = make_texture("meron", geom.grid)
B1 = stray_field(geom.Ms * m1, geom)
B2 = stray_field(geom.Ms * m2, geom)
print(f"initial NRMSE between skyrmion and meron fields: {ap.physics.nrmse(B1, B2).item() * 100:.1f}%")

## 3. Optimize one adversarial pair

`generate_pair` builds both textures, optionally perturbs them, and runs the joint optimization.
By default each charge target is the initial charge (pass `Q1=`, `Q2=` or `mode="difference", q=` to change that).
Always check `charge_constraint_met` — the printout warns if a texture unwound to cheat the field term.

In [ ]:
res = generate_pair("skyrmion", "meron", geom, settings=settings, device=DEVICE)
print(res)
print("charge constraint met:", res.charge_constraint_met)
print(f"mean |B1 - B2| = {res.mean_field_difference_T * 1e3:.4f} mT,  max = {res.max_field_difference_T * 1e3:.4f} mT")

In [ ]:
plotting.plot_history(res);

In [ ]:
plotting.plot_pair(res);

### Holding Ms fixed

With `ms_mode="fixed"` both textures keep Ms = Ms0, so the optimizer cannot compensate the meron's smaller
net moment by inflating its Ms. It still finds an adversarial pair, but the topology term must start
stronger (`beta0=100`); with `beta0=1` it unwinds a texture instead, and the result is flagged.

In [ ]:
fixed = generate_pair("skyrmion", "meron", geom,
                      settings=settings.replace(ms_mode="fixed", beta0=100, log_every=0), device=DEVICE)
print(fixed)

naive = generate_pair("skyrmion", "meron", geom,
                      settings=settings.replace(ms_mode="fixed", beta0=1, log_every=0), device=DEVICE)
print(naive)    # expect a WARNING: the constraint is violated

### Other pairs and charge targets

Any two registered textures (or your own `(nx, ny, 3)` tensors) can be paired. `perturbation` adds
Gaussian noise before optimization; `seed` makes it reproducible.

In [ ]:
other = generate_pair("skyrmion:helicity=0", "fractional:charge=0.25", geom,
                      settings=settings.replace(log_every=0), perturbation=0.01, seed=1, device=DEVICE)
print(other)

diff = generate_pair("meron", "uniform", geom, settings=settings.replace(log_every=0),
                     mode="difference", q=0.5, device=DEVICE)
print(diff)

## 4. Multipole analysis

Why can the field not distinguish the two textures? Far from the sample, the field is set by the
low-order moments of the magnetic source distribution. `multipole_comparison` computes the Cartesian
moments $M_{pq} = \iint x^p y^q\,\rho\,dA$ of each optimized texture (centred on its own centroid,
coordinates normalized to $[-\tfrac12,\tfrac12]$, in units of Ms0) and their relative difference.

* `source="mz"` uses the physical out-of-plane magnetization $M_s m_z$ (surface charge).
* `source="volume"` uses the volume charge $-M_s\nabla\cdot\mathbf m_\parallel$.

In [ ]:
rows = analysis.multipole_comparison(res, order_max=8, source="mz")
print(analysis.format_moment_table(rows))
plotting.plot_multipoles(rows);

In [ ]:
rows_vol = analysis.multipole_comparison(res, order_max=8, source="volume")
print(analysis.format_moment_table(rows_vol))

## 5. k-space localization

The stray field at height $h$ is the source filtered by $e^{-kh}$, so only a low-$k$ band reaches the sensor.
`kspace_localization` bins the radial power spectra of both fields and of their difference, finds the band
holding 90% of the field power, and reports how much of the residual sits inside it. Parseval's identity
(`l2_real` vs `l2_kspace`) is checked as a sanity test.

In [ ]:
loc = analysis.kspace_localization(res)
lo, hi = loc["band"]
print(f"90% of field power in k = {lo:.3f} .. {hi:.3f} rad/nm")
print(f"fraction of residual power inside that band: {loc['frac_residual_in_band'] * 100:.1f}%")
print(f"Parseval check: real-space L2 {loc['l2_real']:.4e}  vs  k-space L2 {loc['l2_kspace']:.4e}")
plotting.plot_kspace(loc);

## 6. Monte Carlo statistical validation

One pair could be luck. The Monte Carlo run draws random texture pairs (random classes, polarities and
perturbations), optimizes each, and collects statistics: the NRMSE distribution, the fraction of pairs below 2%,
the charge-retention error, the Ms ratios, and a breakdown by pair type.

Samples are seeded by `(seed, index)`, so a run is reproducible regardless of `n_workers`.
`N_MC = 20` keeps this cell short; the manuscript uses 100.

In [ ]:
N_MC, WORKERS = 20, 4

mc_cfg = MonteCarloConfig(
    n_samples=N_MC,
    texture_classes=("skyrmion", "meron", "fractional", "uniform"),
    first_texture=None,       # e.g. "skyrmion" to always use a skyrmion as texture 1
    perturbation=0.01,
    mode="anchor",
    charge_grid=0.25,         # snap initial charges to multiples of 1/4 to set targets
    seed=0,
    n_workers=WORKERS,
)
mc_settings = settings.replace(log_every=0)
mc = run_monte_carlo(geom, mc_cfg, mc_settings, device="cpu")

In [ ]:
import json
print(json.dumps(mc.summary(), indent=2))

In [ ]:
for pair_type, stats in mc.by_pair_type().items():
    print(f"{pair_type:28s} n={stats['n']:3d}  mean NRMSE {stats['nrmse_mean'] * 100:6.2f}%")
plotting.plot_mc_summary(mc);

## 7. Monte Carlo across standoff heights

`standoff_sweep` repeats the same seeded ensemble at several heights, so differences between heights reflect the
physics, not sampling noise. Larger standoff means stronger low-pass filtering and easier ambiguity.

In [ ]:
import dataclasses
HEIGHTS = [5, 10, 20, 30, 50]
sweep = standoff_sweep(HEIGHTS, geom, dataclasses.replace(mc_cfg, n_samples=10),
                       mc_settings, progress=False)
for h, r in sweep.items():
    s = r.summary()
    print(f"h = {h:5.1f} nm   NRMSE {s['final_nrmse_mean'] * 100:6.2f} ± {s['final_nrmse_std'] * 100:5.2f}%")
plotting.plot_standoff_sweep(sweep);

## 8. Sensitivity of the skyrmion–meron pair

### Standoff height
The single test-case pair, optimized independently at each height: NRMSE and the mean absolute field difference.

In [ ]:
rows_h = analysis.standoff_sensitivity(np.linspace(5, 50, 10), "skyrmion", "meron", geom,
                                       settings=settings.replace(log_every=0), device=DEVICE)
print(analysis.format_rows(rows_h))

h = [r["height_nm"] for r in rows_h]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(h, [r["nrmse"] * 100 for r in rows_h], "o-"); a1.set_xlabel("standoff (nm)"); a1.set_ylabel("NRMSE (%)")
a2.semilogy(h, [r["mean_abs_dB_T"] * 1e3 for r in rows_h], "s-"); a2.set_xlabel("standoff (nm)"); a2.set_ylabel("mean |B1 − B2| (mT)")
plt.tight_layout()

### Grid resolution
Same physical sample (64 nm), finer grids. The result should not depend on discretization.
128×128 is slow on CPU; add it if you have a GPU or time.

In [ ]:
rows_g = analysis.grid_sensitivity([(32, 32), (64, 64)], "skyrmion", "meron", geom,
                                   settings=settings.replace(log_every=0), device=DEVICE)
print(analysis.format_rows(rows_g))

### Loss weights
Any `OptimizerSettings` field can be swept, e.g. the smoothness weight γ.

In [ ]:
rows_w = analysis.weight_sensitivity("gamma", [1e-4, 1e-3, 1e-2], "skyrmion", "meron", geom,
                                     settings=settings.replace(log_every=0), device=DEVICE)
print(analysis.format_rows(rows_w))

## 9. Save results

`PairResult.save` writes `pair.npz` (textures, fields in T, Ms, loss history) and `summary.json`;
`PairResult.load` reads them back. `MonteCarloResult.save` writes `samples.csv` and `summary.json`.

In [ ]:
res.save("results/notebook_pair")
mc.save("results/notebook_mc")
back = ap.PairResult.load("results/notebook_pair")
print(back)